## 简介与参数说明

TodoListMiddleware 中间件赋予 Agent **任务规划** 和 **追踪进度** 的能力，可以应对复杂的多步任务。
比如，当一个大任务需要被拆解为 3 个以上的子任务，且前面的步骤是后面步骤的前提时，如果不列 Todo 列表，大模型在执行到第 3 步时，很容易忘记自己最初的目标，或者在工具返回大量报错信息后“应激”，直接跳过验证去回答用户。

TodoListMiddleware **不会强制模型一定创建计划**：它会向 Agent 注册 `write_todos` 工具，并注入一段指导模型何时、如何使用 Todo 的系统提示词。是否调用该工具仍由模型决定；因此简单问答或模型判断为无需规划的任务，最终状态中可能没有 `todos`。

当模型调用 `write_todos` 后，待办列表会写入 Agent 的运行状态 `todos`，让后续模型调用能够看到当前计划和进度。

### `write_todos` 的状态语义

- 每个事项固定为 `{"content": "任务描述", "status": "pending" | "in_progress" | "completed"}`。
- 一次调用会**整体替换**当前 Todo 列表，而不是向列表追加一项；更新时应把仍需保留的事项一起传回。
- 同一轮模型输出中最多只能调用一次 `write_todos`。多个并行调用都会尝试替换同一个列表，LangChain 会将其拒绝为错误。
- Todo 是模型维护的计划与可视化进度，不是自动执行器：它不会自动运行任务、验证完成状态，也不会表达 DAG 依赖关系。

```
你的任务是否需要拆解？
├─ 否（比如：问答、翻译、单次函数调用）→ ✖ 不需要，使用 Todo 反而增加 token 与工具调用开销
└─ 是（比如：写一个包含多文件的工程）
    ├─ 步骤是否多变且需要应对失败？
    │   ├─ 否（步骤完全固定，如 A->B->C）→ ✖ 传统的 LangGraph 线性节点即可
    │   └─ 是（AI 需要边做边调计划）→ 引入 TodoListMiddleware
```

如果把普通 Agent 比作“想到哪写到哪”的实习生，那么引入了 TodoListMiddleware 的 Agent 更像是“先列 CheckList、执行中持续更新”的执行者。

### 典型场景
- 任务链路长、步骤多，且有严格的先后依赖关系
- 需要在前端 UI 界面实时展示 Agent 的执行进度（下方会用流式 `updates` 演示）
- 多个工具协作完成代码修复、资料调研、数据处理等任务

To-do list 的创建和维护是通过调用 `write_todos` 工具实现的。

### 参数说明
1. **system_prompt** — 自定义指导 Todo 列表使用的提示词。
   不提供则使用内置提示词，通常不必提供。

2. **tool_description** — 自定义 `write_todos` 工具的描述信息。
   不提供则使用内置描述，通常不必提供。

部分示例代码：`TodoListMiddlewareDemo`

其中测试文件的约定：
1. pytest 会扫描目录下所有以 `test_` 开头或以 `_test` 结尾的文件，视为测试文件。
2. 然后执行测试文件中所有以 `test` 开头的函数。
3. 执行出错会打印到控制台。
4. 测试函数调用 `my_add.py` 中的 `add`；结果不符合预期时会抛出异常。


In [2]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)


In [4]:
from langchain.tools import tool
from pathlib import Path
import subprocess

WORKSPACE = Path("TodoListMiddlewareDemo")

@tool
def list_files(path: str = ".") -> str:
    """
    列出工作区指定目录下的文件和子目录。path 只能是相对路径。

    Args:
        path: 工作区下的相对路径，一定指向目录，默认为.，表示工作区根路径，不能访问工作区外的目录
    """
    target = (WORKSPACE / path).resolve()
    workspace_root = WORKSPACE.resolve()

    if not str(target).startswith(str(workspace_root)):
        return "错误：只允许访问工作区内的目录。"

    if not target.exists():
        return f"错误：目录不存在：{path}"

    if not target.is_dir():
        return f"错误：不是目录：{path}"

    items = sorted(target.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    if not items:
        return f"目录为空：{path}"

    lines = []
    for item in items:
        rel = item.relative_to(workspace_root)
        kind = "[DIR]" if item.is_dir() else "[FILE]"
        lines.append(f"{kind} {rel.as_posix()}")

    return "\n".join(lines)


@tool
def read_file(path: str) -> str:
    """
    读取工作区中的文本文件内容。path 只能是相对路径。

    Args:
        path: 工作区内的文件名
    """
    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "错误：只允许读取工作区内的文件。"
    if not file_path.exists():
        return f"错误：文件不存在：{path}"
    return file_path.read_text(encoding="utf-8")


@tool
def write_file(path: str, content: str) -> str:
    """
    写入工作区中的文本文件。path 只能是相对路径。

    Args:
        path: 工作区内的文件名
        content: 写入文件的内容
    """
    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "错误：只允许写入工作区内的文件。"
    file_path.write_text(content, encoding="utf-8")
    return f"已写入文件：{path}"


@tool
def run_tests() -> str:
    """
    在工作区运行 pytest -q，并返回输出。
    不接收任何参数，返回格式为
    returncode=0|1
    STDOUT:
    STDERR:
    """
    try:
        result = subprocess.run(
            ["pytest", "-q"],
            cwd=str(WORKSPACE),
            capture_output=True,
            text=True,
            timeout=20,
        )
        return (
            f"returncode={result.returncode}\n\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )
    except Exception as e:
        return f"运行测试失败：{e}"


### 重置可复现的故障样例

代码修复 Agent 会修改 `my_add.py`。为了让每次从头运行笔记本时都能稳定演示“测试失败 -> 修复 -> 复测”，在调用 Agent 前先把目标文件重置为一个故意错误的实现。


In [ ]:
# 仅用于教学：每次运行前恢复一个会导致 pytest 失败的实现。
BUGGY_ADD_SOURCE = '''def add(a: int, b: int) -> int:
    """返回两个整数的和。"""
    return a - b
'''

demo_file = WORKSPACE / "my_add.py"
demo_file.write_text(BUGGY_ADD_SOURCE, encoding="utf-8")
print(f"已重置故障样例：{demo_file}")


### 实时读取 Todo 状态

这里用一次 `stream(..., stream_mode=["updates", "values"], version="v2")` 运行 Agent：

- `updates` 是节点产生的增量状态，用它捕获 `write_todos` 写入的 `todos` 并立即刷新终端或前端 UI。
- `values` 是每一步后的完整状态；保留最后一次的值作为 `final_state`，供下一单元读取最终 Todo 和回答。

这样不会先 `invoke()` 再额外运行一次流式调用，避免为同一任务支付两次模型调用成本。


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain.messages import HumanMessage

# 1. 初始化 Agent
agent = create_agent(
    model=model,
    tools=[list_files, read_file, write_file, run_tests],
    middleware=[TodoListMiddleware()],
    system_prompt=(
        "你是一个代码修复助手。遇到多步骤任务时，先使用 write_todos 制定待办事项；"
        "然后读取文件、修复代码并运行测试。工作全部在工作区下进行。"
    ),
)

# 2. 一次流式运行：实时显示 Todo 更新，同时保留最终状态。
print("正在执行 Agent 任务...")
final_state = None
last_todos = None

for chunk in agent.stream(
    {
        "messages": [
            HumanMessage(content="请测试并修复工作区下 my_add.py 文件中的代码")
        ]
    },
    stream_mode=["updates", "values"],
    version="v2",
):
    if chunk["type"] == "updates":
        # write_todos 在工具节点返回 Command(update={"todos": ...})；
        # 因此可从节点增量中立即取到最新列表，推送给终端或前端。
        for node_name, node_update in chunk["data"].items():
            todos = node_update.get("todos") if isinstance(node_update, dict) else None
            if todos is not None and todos != last_todos:
                print(f"\n[来自 {node_name} 节点的 Todo 更新]")
                for index, todo in enumerate(todos, 1):
                    print(f"{index}. [{todo['status']}] {todo['content']}")
                last_todos = todos
    elif chunk["type"] == "values":
        # 最后一个 values 即 Agent 结束后的完整图状态。
        final_state = chunk["data"]

if final_state is None:
    raise RuntimeError("Agent 未返回最终状态")


In [ ]:
# 3. 直观展示中间件产生的数据结果
print("\n" + "=" * 20 + " 1. 最终 TODO 列表 " + "=" * 20)
# TodoListMiddleware 的状态结构固定为：
# {"content": "任务描述", "status": "pending" | "in_progress" | "completed"}
todos = final_state.get("todos", [])
if todos:
    for index, todo in enumerate(todos, 1):
        print(f"{index}. [{todo['status']}] {todo['content']}")
else:
    print("未检测到待办事项（可能 Agent 认为任务足够简单，未触发 write_todos 工具）")

print("\n" + "=" * 20 + " 2. Agent 最终修复回复 " + "=" * 20)
# 获取对话历史中的最后一条消息（即 Agent 的最终总结）。
if final_state.get("messages"):
    print(final_state["messages"][-1].content)
